# Q//Horizon 2026: Quantum vs. Classical Pulsar Classification (UCI HTRU2)
**End-to-End Reproducible Benchmark Notebook**

This notebook verifies our leakage-free HTRU2 data pipeline, displays our pre-computed 10-seed benchmark tables (`results/*.csv`), renders the 300-DPI error-bar figures (`results/*.png`), and summarizes the physical and statistical takeaways for each experiment.


In [ ]:
import pandas as pd
from IPython.display import Image, display

from src.config import FAST_MODE, RESULTS_DIR, SEEDS, N_TUNING_CONFIGS
from src.data import get_locked_splits, sample_and_preprocess
from src.plot_results import main as generate_plots

print(f"FAST_MODE={FAST_MODE} | Active seeds={len(SEEDS)} | Tuning budget={N_TUNING_CONFIGS}")
splits = get_locked_splits()
sample = sample_and_preprocess(splits, n_train=500, seed=100)
print(f"Locked splits -> Pool: {splits['X_pool'].shape}, Val: {splits['X_val'].shape}, Test: {splits['X_test'].shape}")
print(f"Train positive pulsar ratio: {(sample['y_train'] == 1).mean():.4f} | Test positive ratio: {(sample['y_test'] == 1).mean():.4f}")
generate_plots()


## 1. Main Benchmark ($N_{\text{train}} = 500$, 10 Seeds)
All tunable classical and quantum models share an identical 20-configuration validation random-search budget (`val_pr_auc`), with decision thresholds selected strictly on `y_val`.


In [ ]:
df_main = pd.read_csv(RESULTS_DIR / "main_benchmark.csv")
display(df_main.groupby("model", sort=False)[["test_pr_auc", "test_f1", "test_mcc", "test_acc", "n_params"]].agg(["mean", "std"]).round(4))
display(Image(filename=str(RESULTS_DIR / "main_benchmark.png")))


**Why did this happen?**
* **The 90.83% Accuracy Trap:** `Dummy` achieves `0.9083` accuracy by predicting all candidates as radio-frequency interference (`0`), yet scores `0.0917` PR-AUC and `0.0000` MCC?proving why PR-AUC and MCC are mandatory.
* **Kernel vs. Variational Quantum:** `Q2-QuantumKernel (8q)` (`0.9023 ? 0.0230` PR-AUC) outperforms `RandomForest` (`0.9015 ? 0.0156`) via convex SVM optimization in Hilbert space, whereas gradient-trained `Q1-VQC (8q Ring L=2)` (`0.7800 ? 0.1437`) suffers high seed variance under non-convex optimization.


## 2. Gold A: Entanglement Topology & Depth Ablation
Testing unentangled (`none`), nearest-neighbor (`ring`), and domain-paired (`bipartite`, linking Pulse Profile wires `0?3` to DM-SNR wires `4?7`) circuits across depths $L \in \{1, 2, 4\}$.


In [ ]:
df_a = pd.read_csv(RESULTS_DIR / "gold_a_entanglement.csv")
display(df_a.groupby("model", sort=False)[["test_pr_auc", "test_f1", "test_mcc", "init_grad_norm"]].agg(["mean", "std"]).round(4))
display(Image(filename=str(RESULTS_DIR / "gold_a_entanglement.png")))


**Why did this happen?**
* Domain-structured `VQC_bipartite_L4` (`0.8990 ? 0.0113` PR-AUC) and `VQC_bipartite_L2` (`0.8981 ? 0.0057` PR-AUC, `0.8334 ? 0.0074` MCC) achieve the highest VQC scores by coupling integrated pulse-profile features with DM-SNR features.
* However, unentangled `VQC_none_L1` (`0.8945 ? 0.0010` PR-AUC) achieves nearly identical performance with 10x smaller variance, showing that smooth tabular decision boundaries rely primarily on single-qubit feature rotations rather than multi-qubit entanglement.


## 3. Gold B: Quantum Encoding & Radial Norm Restoration
Comparing 8-qubit angle encoding (1 feature per qubit) against 3-qubit amplitude compression ($2^3 = 8$ features) and 4-qubit amplitude encoding with the discarded $L_2$ vector norm restored on wire 3.


In [ ]:
df_b = pd.read_csv(RESULTS_DIR / "gold_b_encoding.csv")
display(df_b.groupby("model", sort=False)[["test_pr_auc", "test_f1", "test_mcc", "n_params"]].agg(["mean", "std"]).round(4))
display(Image(filename=str(RESULTS_DIR / "gold_b_encoding.png")))


**Why did this happen?**
* Normalizing an 8-feature vector onto unit-norm quantum amplitudes (`3q_Amplitude_L5`) collapses PR-AUC to `0.6185 ? 0.0473` because extreme pulsar skewness and kurtosis live in the radial magnitude $\Vert{}x\Vert{}_2$.
* Re-injecting $\Vert{}x\Vert{}_2$ as a rotation angle on wire 3 (`4q_AmplitudePlusNorm_L4`) recovers PR-AUC to `0.8431 ? 0.0151` (`+0.2246` PR-AUC gain) using half as many qubits as angle encoding.


## 4. Gold C & Table D: Sample Efficiency Sweep & Finite Shot Noise


In [ ]:
df_c = pd.read_csv(RESULTS_DIR / "gold_c_sweep.csv")
display(df_c.groupby(["n_train", "model"], sort=False)[["test_pr_auc", "test_mcc"]].agg(["mean", "std"]).round(4))
display(Image(filename=str(RESULTS_DIR / "gold_c_sweep.png")))

df_d = pd.read_csv(RESULTS_DIR / "table_d_shots.csv")
display(df_d.groupby("shots", sort=False)[["test_pr_auc", "test_f1", "test_mcc"]].agg(["mean", "std"]).round(4))
display(Image(filename=str(RESULTS_DIR / "table_d_shots.png")))


**Why did this happen?**
* **Low-Data Regime:** At $N=100$ (~9 positive pulsars), `Q2-QuantumKernel (8q)` (`0.8884 ? 0.0181` PR-AUC) matches `RBF-SVM` (`0.8900 ? 0.0236`), while `Q1-VQC` stabilizes only once $N=1000$ (`0.8960 ? 0.0045`).
* **Shot Noise:** Finite measurement sampling degrades PR-AUC by only `0.0039` at `2,000` shots (`0.7761` vs. `0.7800` analytic) and `0.0189` at `500` shots (`0.7611`), showing strong resilience down to moderate NISQ shot budgets.
